# 11 — Spring Boot in Production

**What you'll learn:**

- What Spring Boot adds on top of plain Spring — and what `@SpringBootApplication` actually does
- **Auto-configuration** — how Boot wires beans based on what's on the classpath
- **Externalised configuration** in depth — the property source precedence chain
- `@ConfigurationProperties` with validation
- **Actuator** — health, metrics, env, beans, and writing custom endpoints
- **Structured logging** — Logback, the SLF4J facade, JSON output for production
- The **test slices** — `@SpringBootTest`, `@WebMvcTest`, `@DataJpaTest`, `@JsonTest`
- Mocking with `@MockitoBean` (formerly `@MockBean`)
- **Testcontainers** — real databases in your integration tests, ephemeral and fast
- **Messaging with `spring-kafka`** — producer template, `@KafkaListener` consumers, serialisation
- JMS and Spring AMQP (RabbitMQ) in one paragraph each
- **Packaging** — the fat JAR layout and the layered JAR for Docker
- **Container deployment** — `Dockerfile` patterns and **Cloud Native Buildpacks** (`spring-boot:build-image`)

This notebook is what turns a Spring service into something you can actually deploy and run reliably. Notebooks 08, 09, and 10 made an application; this one makes it *operable*.

## What Spring Boot adds

Plain Spring is a fantastic dependency-injection container, but it leaves you to configure every part of the runtime — pick a servlet container, set up a `DispatcherServlet`, register the JSON converters, wire the data source. **Spring Boot** is an opinionated meta-framework on top of Spring that does that work for you.

Three things Boot brings to the table:

1. **Auto-configuration** — looks at what's on your classpath and wires sensible defaults.
2. **Starter dependencies** — curated bundles like `spring-boot-starter-web` that pull in everything you need for a category of feature.
3. **Embedded servers** — Tomcat (or Jetty/Undertow) packaged inside your JAR, so deployment is `java -jar`.

The entry point is one annotation and one `main` method:

In [ ]:
@SpringBootApplication
public class App {
    public static void main(String[] args) {
        SpringApplication.run(App.class, args);
    }
}

**`@SpringBootApplication` is three annotations in one:**

- `@SpringBootConfiguration` — marks the class as a `@Configuration`
- `@ComponentScan` — scan this package and all sub-packages for stereotypes
- `@EnableAutoConfiguration` — turn on auto-configuration

That's the whole entry point. From here, Boot finds your `@Component`s, applies its conditional auto-configurations, starts the embedded server, and your service is up.

## Auto-configuration — how it works

Boot ships dozens of auto-configuration classes. Each one is a `@Configuration` that's only *active* when certain conditions hold — usually "this library is on the classpath" or "no user-defined bean of this type exists yet".

The mechanism is built from a family of `@Conditional` annotations:

| Annotation | Triggers when... |
|---|---|
| `@ConditionalOnClass(X.class)` | class `X` is on the classpath |
| `@ConditionalOnMissingBean(X.class)` | no bean of type `X` is already defined |
| `@ConditionalOnProperty("foo.enabled")` | property is set (and truthy) |
| `@ConditionalOnWebApplication` | running inside a servlet web context |

A typical auto-configuration reads like this:

In [ ]:
@AutoConfiguration
@ConditionalOnClass(DataSource.class)
@EnableConfigurationProperties(DataSourceProperties.class)
public class DataSourceAutoConfiguration {

    @Bean
    @ConditionalOnMissingBean
    public DataSource dataSource(DataSourceProperties props) {
        return /* HikariCP-backed DataSource built from props */;
    }
}

Reading that as English: *"if `DataSource` is on the classpath, and the user hasn't defined their own `DataSource` bean, register a Hikari one built from `spring.datasource.*` properties."* You override Boot's defaults simply by declaring your own bean — your bean wins, and the auto-configured one is skipped.

To inspect what auto-configurations fired (and which ones didn't), pass `--debug` on the command line, or hit the `/actuator/conditions` endpoint. Boot prints a *positive matches / negative matches* report — invaluable when you're debugging "why isn't this bean being created".

## Externalised configuration — the precedence chain

The same code should run in dev, staging, and prod with different values. Boot reads properties from many sources and merges them, with later sources overriding earlier ones. The order, from lowest to highest precedence:

```text
   1. @PropertySource on a @Configuration class
   2. application.properties or application.yml on the classpath
   3. application-{profile}.yml when that profile is active
   4. OS environment variables           SPRING_DATASOURCE_URL
   5. Java system properties             -Dspring.datasource.url=...
   6. Command-line arguments             --spring.datasource.url=...
```

Higher numbers win. In practice, this means you commit safe defaults in `application.yml`, override per environment with env vars, and override per run with command-line flags.

In [ ]:
# application.yml — safe defaults committed to the repo
server:
  port: 8080
spring:
  application:
    name: checkout-service
  datasource:
    url: jdbc:postgresql://localhost:5432/checkout
    username: app
    password: dev-password

# application-prod.yml — overlays activated when spring.profiles.active=prod
spring:
  datasource:
    url: ${DB_URL}
    username: ${DB_USER}
    password: ${DB_PASSWORD}
logging:
  level:
    root: WARN

**Notation matters.** Spring normalises property names so `spring.datasource.url` in YAML, `SPRING_DATASOURCE_URL` as an env var, `--spring.datasource.url=...` on the command line, and `-Dspring.datasource.url=...` as a JVM flag all bind to the same property. This is what makes the same JAR portable across local dev, Docker, and Kubernetes without code changes.

## `@ConfigurationProperties` with validation

For non-trivial configuration, the right shape is a **typed record** annotated `@ConfigurationProperties`. You group a prefix's worth of properties into one bean, get IDE autocomplete, and can validate at startup.

In [ ]:
import jakarta.validation.constraints.*;
import org.springframework.boot.context.properties.*;
import org.springframework.validation.annotation.Validated;

@ConfigurationProperties(prefix = "checkout")
@Validated
public record CheckoutProps(
    @NotBlank
    String currency,

    @Positive @Max(10_000_00)
    long maxAmountCents,

    @NotNull
    Duration timeout,

    Retry retry
) {
    public record Retry(@Min(0) @Max(10) int maxAttempts, Duration backoff) {}
}

// in your @SpringBootApplication class, or any @Configuration:
@EnableConfigurationProperties(CheckoutProps.class)

**The startup contract is strict.** If a required property is missing or fails validation, the application **fails to start** with a clear error. This is exactly what you want — better a noisy boot failure than a silent misconfiguration that surfaces an hour into production traffic.

`Duration` is parsed from the standard ISO-8601 form (`PT30S`, `PT5M`) or the friendlier `30s` / `5m`. `DataSize` parses things like `10MB`. These conversions are built in.

## Actuator — health, metrics, and introspection

Add `spring-boot-starter-actuator` and you get a set of HTTP endpoints exposing the runtime's vitals. The defaults are conservative — `/actuator/health` is the only one exposed by default — and you opt into the rest.

In [ ]:
# application.yml
management:
  endpoints:
    web:
      exposure:
        include: health, info, metrics, env, beans, conditions, mappings
  endpoint:
    health:
      show-details: when-authorized
  metrics:
    distribution:
      percentiles-histogram:
        http.server.requests: true

The endpoints you'll use most:

| Endpoint | What it shows |
|---|---|
| `/actuator/health` | overall health + per-component (DB, Redis, Kafka, ...) |
| `/actuator/info` | static info (app name, version, build time) |
| `/actuator/metrics` | every Micrometer metric the app records |
| `/actuator/env` | every property and its source (great for "why is this value...") |
| `/actuator/beans` | the entire `ApplicationContext` bean graph |
| `/actuator/conditions` | auto-configuration matches and non-matches |
| `/actuator/mappings` | every HTTP route the app handles |
| `/actuator/loggers` | view + **change** log levels at runtime |

`/actuator/health` is the endpoint your load balancer or Kubernetes liveness probe polls. Spring auto-generates separate `/health/liveness` and `/health/readiness` groups for Kubernetes-style probes.

In [ ]:
// custom health indicator — runs as part of /actuator/health
@Component
public class KafkaHealthIndicator implements HealthIndicator {
    private final AdminClient admin;

    public KafkaHealthIndicator(AdminClient admin) { this.admin = admin; }

    @Override
    public Health health() {
        try {
            admin.listTopics().names().get(2, TimeUnit.SECONDS);
            return Health.up().build();
        } catch (Exception e) {
            return Health.down(e).build();
        }
    }
}

**Metrics** flow through **Micrometer**, Boot's metrics abstraction. Boot publishes JVM metrics, HTTP server metrics, JDBC connection pool metrics, Kafka client metrics, and more — all out of the box. You can scrape them via `/actuator/prometheus` (with the Prometheus registry on the classpath) or push to any of Micrometer's supported backends (Datadog, New Relic, CloudWatch, etc.).

## Logging — Logback, SLF4J, structured output

Boot ships **Logback** as the default logging implementation, behind the **SLF4J** facade. Every Spring class uses SLF4J; you do the same.

In [ ]:
import org.slf4j.*;

@Service
public class CheckoutService {
    private static final Logger log = LoggerFactory.getLogger(CheckoutService.class);

    public void process(Order order) {
        log.info("processing order id={} amount={}", order.id(), order.amount());
        try {
            // ...
        } catch (Exception e) {
            log.error("order {} failed", order.id(), e);   // exception goes as LAST arg
            throw e;
        }
    }
}

**Two pieces of style discipline** save you from a lot of operational pain:

- Use `{}` placeholders, not `+` concatenation. SLF4J skips formatting at disabled levels — concatenation always pays the cost.
- Pass the exception as the **last** argument (no placeholder for it). SLF4J detects it and prints the stack trace.

Configure levels with `logging.level.<package>=DEBUG` in `application.yml`, or change them at runtime via `/actuator/loggers`. Use **structured JSON logging** in production — Logback's `LogstashEncoder` (from the `logstash-logback-encoder` dependency) turns every log event into a JSON line that log aggregators (Splunk, Elasticsearch, Loki, Datadog) can index and query.

## Testing — the test slices

Boot ships *slice* annotations that load only the part of the `ApplicationContext` you need to test. They start much faster than a full context and isolate the layer under test.

| Annotation | Loads |
|---|---|
| `@SpringBootTest` | the full context — true integration tests |
| `@WebMvcTest(Controller.class)` | only the web layer (MVC, JSON, security) |
| `@DataJpaTest` | only JPA repositories + an in-memory database |
| `@JsonTest` | only the Jackson configuration |
| `@RestClientTest(Gateway.class)` | only the HTTP client config + a `MockRestServiceServer` |

In [ ]:
// full-context integration test — slow but realistic
@SpringBootTest(webEnvironment = SpringBootTest.WebEnvironment.RANDOM_PORT)
class CheckoutAppTests {

    @Autowired TestRestTemplate rest;

    @Test
    void healthIsUp() {
        var response = rest.getForEntity("/actuator/health", String.class);
        assertTrue(response.getStatusCode().is2xxSuccessful());
    }
}

// web-slice test — only the controller, with services mocked
@WebMvcTest(CheckoutController.class)
class CheckoutControllerTests {

    @Autowired MockMvc mockMvc;
    @MockitoBean CheckoutService service;     // replaces the real bean with a Mockito mock

    @Test
    void returnsCreatedOnPost() throws Exception {
        when(service.process(any())).thenReturn(new Order(1L, 100));

        mockMvc.perform(post("/api/orders")
                .contentType(MediaType.APPLICATION_JSON)
                .content("{\"amount\": 100}"))
            .andExpect(status().isCreated())
            .andExpect(jsonPath("$.id").value(1));
    }
}

// data-slice test — repository against an in-memory H2 by default
@DataJpaTest
class UserRepositoryTests {

    @Autowired UserRepository repo;
    @Autowired TestEntityManager em;

    @Test
    void findsByEmail() {
        em.persist(new User("a@b.com", "alice"));
        assertTrue(repo.findByEmail("a@b.com").isPresent());
    }
}

**`@MockitoBean`** (replaces the older `@MockBean` in Boot 3.4+) substitutes a Mockito mock for a bean in the application context. Use it in slice tests to isolate the layer under test from its collaborators.

Two patterns worth internalising:

- Default to **slice tests**. They're 10× faster than a full `@SpringBootTest` and force you to write decoupled code.
- Reach for `@SpringBootTest` only for **end-to-end checks** — startup, security wiring, full request roundtrips.

## Testcontainers — real databases, ephemeral

In-memory H2 is fine for trivial repositories, but the moment you use Postgres-specific syntax (`JSONB`, full-text search, partial indexes), H2 disagrees and your tests give false confidence. **Testcontainers** spins up a real Postgres in Docker for the duration of your test class, then tears it down. Tests run against the same database technology as production.

In [ ]:
@SpringBootTest
@Testcontainers
class PostRepositoryIntegrationTests {

    @Container
    static PostgreSQLContainer<?> postgres = new PostgreSQLContainer<>("postgres:16");

    @DynamicPropertySource
    static void wireProps(DynamicPropertyRegistry r) {
        r.add("spring.datasource.url", postgres::getJdbcUrl);
        r.add("spring.datasource.username", postgres::getUsername);
        r.add("spring.datasource.password", postgres::getPassword);
    }

    @Autowired PostRepository repo;

    @Test
    void fullTextSearchReturnsExpectedHits() {
        // your real Postgres-specific SQL works here
    }
}

Spring Boot 3.1+ has **first-class Testcontainers support** via `@ServiceConnection` — drop the annotation on a container bean and Boot wires the right `spring.datasource.*` (or `spring.kafka.*`, or `spring.redis.*`) properties automatically. Testcontainers supports Postgres, MySQL, MongoDB, Kafka, Redis, RabbitMQ, Elasticsearch, LocalStack (for AWS), and dozens of others. It's the modern default for integration testing on the JVM.

## Messaging with `spring-kafka`

Kafka is the dominant message bus in modern back ends. The `spring-kafka` library gives you a `KafkaTemplate` for producing and the `@KafkaListener` annotation for consuming.

In [ ]:
# application.yml
spring:
  kafka:
    bootstrap-servers: localhost:9092
    producer:
      key-serializer: org.apache.kafka.common.serialization.StringSerializer
      value-serializer: org.springframework.kafka.support.serializer.JsonSerializer
    consumer:
      group-id: checkout
      key-deserializer: org.apache.kafka.common.serialization.StringDeserializer
      value-deserializer: org.springframework.kafka.support.serializer.JsonDeserializer
      properties:
        spring.json.trusted.packages: "com.example.events"

In [ ]:
// the event — a record is the ideal shape
public record OrderPlaced(long orderId, long customerId, long amountCents) {}

// producer — inject KafkaTemplate
@Service
public class OrderPublisher {
    private final KafkaTemplate<String, OrderPlaced> kafka;

    public OrderPublisher(KafkaTemplate<String, OrderPlaced> kafka) {
        this.kafka = kafka;
    }

    public void publish(OrderPlaced event) {
        kafka.send("orders.placed", String.valueOf(event.orderId()), event);
    }
}

// consumer — one method per topic
@Component
public class OrderConsumer {
    private static final Logger log = LoggerFactory.getLogger(OrderConsumer.class);

    @KafkaListener(topics = "orders.placed", groupId = "fulfilment")
    public void on(OrderPlaced event) {
        log.info("processing order {}", event.orderId());
        // do the work
    }
}

A few production-grade details to know:

- **Keys matter** — Kafka partitions by key. Use a stable key (the order id, the user id) so all events for the same entity land on the same partition and stay ordered.
- **Consumer group** — multiple instances of your service sharing the same `groupId` split partitions between them. That's how you scale horizontally.
- **Error handling** — `spring-kafka` ships a default `DefaultErrorHandler` that retries with backoff and then sends poison-pill messages to a *dead-letter topic*. Wire one up if you want anything more sophisticated than "fail and skip".
- **Acks and idempotence** — for at-least-once delivery semantics, set `acks=all` on the producer and `enable.idempotence=true`. The defaults in modern `spring-kafka` are reasonable but always verify against your durability requirements.

## JMS and Spring AMQP — in one paragraph each

**JMS** (Java Message Service) is the older JVM messaging spec, typically used with brokers like ActiveMQ or IBM MQ. Spring's `JmsTemplate` and `@JmsListener` mirror the Kafka shapes. JMS is the right choice when you're integrating with an existing enterprise message bus or need transactional messaging tightly coupled to a JTA transaction manager.

**Spring AMQP** is the Spring abstraction over the AMQP 0-9-1 protocol — almost always meaning **RabbitMQ** in practice. The shapes are again parallel: `RabbitTemplate` for producing, `@RabbitListener` for consuming, with exchanges, queues, and bindings configured as beans. RabbitMQ shines for *task queues* with complex routing (work distribution, RPC patterns, fanout); Kafka shines for *event streams* with replay and long retention.

## Packaging — the fat JAR

Boot packages your application as an **executable fat JAR** (also called *uber JAR*) — your code plus every dependency plus an embedded Tomcat plus a special launcher, all in one file. You run it with `java -jar app.jar`. No external app server, no `WEB-INF` directory.

```text
   app.jar
   +-- BOOT-INF/
   |     +-- classes/    <- your compiled code + resources
   |     +-- lib/        <- every dependency JAR
   |
   +-- META-INF/
   |     +-- MANIFEST.MF <- Main-Class: org.springframework.boot.loader.JarLauncher
   |
   +-- org/springframework/boot/loader/  <- the launcher
```

Build it with `mvn package` or `./gradlew bootJar`. The output is one self-contained artifact you can copy anywhere with a JDK.

## Container deployment

Two ways to put your Boot app in a container, both mainstream:

In [ ]:
# Dockerfile — the explicit approach
FROM eclipse-temurin:21-jre-alpine
WORKDIR /app
COPY target/app.jar app.jar
EXPOSE 8080
ENTRYPOINT ["java", "-jar", "app.jar"]

That works, but it copies the whole fat JAR as one Docker layer — change one line of code and every byte of every dependency gets re-pushed. The fix is the **layered JAR**, which Boot enables by default since 2.3:

In [ ]:
# Dockerfile — using Boot's layered jar (deps cached separately from app code)
FROM eclipse-temurin:21-jre-alpine AS extractor
WORKDIR /extract
COPY target/app.jar app.jar
RUN java -Djarmode=layertools -jar app.jar extract

FROM eclipse-temurin:21-jre-alpine
WORKDIR /app
COPY --from=extractor /extract/dependencies/         ./
COPY --from=extractor /extract/spring-boot-loader/   ./
COPY --from=extractor /extract/snapshot-dependencies/ ./
COPY --from=extractor /extract/application/          ./
ENTRYPOINT ["java", "org.springframework.boot.loader.launch.JarLauncher"]

Each layer is a separate `COPY`, so unchanged dependencies stay cached. Application code changes only re-push the small `application/` layer.

**Even simpler**: skip the `Dockerfile` entirely and use **Cloud Native Buildpacks**, built into Boot:

```bash
mvn spring-boot:build-image                        # Maven
./gradlew bootBuildImage                           # Gradle
```

That produces an optimised, layered, secure OCI image with no Dockerfile to maintain. Boot uses the [Paketo](https://paketo.io) buildpacks under the hood, which pick a JDK, apply best-practice JVM settings, and rebuild cleanly when only your source changes. For most services, **`bootBuildImage` is the right default**.

## What's next

You have a Spring service you can deploy. Boot's auto-configuration set up the runtime, `@ConfigurationProperties` made it environment-portable, Actuator made it observable, the test slices made it test-able, Testcontainers made the tests *realistic*, `spring-kafka` plugged it into the message bus, and Buildpacks made it a container.

Notebook 12 — the final notebook — turns one Spring service into a **distributed system** with **Spring Cloud**. We'll cover the **config server** for centralised configuration, **service discovery** with Eureka, **Spring Cloud Gateway** as the API edge, **circuit breakers** with Resilience4j, **distributed tracing** via Micrometer + OpenTelemetry, and **Spring Cloud Stream** — the higher-level binder abstraction on top of Kafka and RabbitMQ.